# CASSETTE
- segment MIDI both on bar & on beat level: `pretty_midi`
- extract weighted pitch class profile for each of the segments of MIDI file
    - extract notes sounding between start & end time
    - compute, for each note, product of MIDI velocity & proportion of segment during which note sounds
    - sums product over all notes in same pitch class
    - normalize weighted pitch class profile by dividing each element by total sum of all its elements (makes feature invariant to total loudness & duration of notes in segment)
- find best matching chord for each segment, by assigning chord template that's most similar to normalized weighted pitch class profile of segment
    - vocabulary of 25 chords (24 maj/min chords + no-chord symbol)
    - chord template is 12-dim vector
    - similar score using Pardo & Birmingham $S = P - (N + M)$ where $P$ = sum of weights of pitch classes of bar that match a template element, $N$ = sum of weights of pitch classes of bar that do not match a template element, $M$ = count of template elements not matched by any note
    - chord with highest template similarity score is assigned
    - if score <= -3, algorithm assigned no-chord
    - if multiple templates have same similarity score, selects template whose root pitch has greatest weight in segment's pitch class profile

##Install and import packages, set up Drive for Dataset

In [ ]:
# run in base_environment
# !pip install pretty_midi mir_eval mirdata pyfluidsynth

# Packages
import pretty_midi
import mirdata
import mir_eval
import numpy as np
import pandas as pd
from pathlib import Path

# Our functions
import utils as u

# For plotting
import mir_eval.display
import librosa.display
import matplotlib.pyplot as plt

# For audio display
from IPython.display import Audio

## Initialize loader, download and load data using mirdata





For this project we will use SLAKH datset, a synthesized version of the LAKH dataset:
<blockquote>
Manilow, Ethan, Gordon Wichern, Prem Seetharaman, and Jonathan Le Roux. "Cutting music source separation some Slakh: A dataset to study the impact of training data quality and quantity." In 2019 IEEE Workshop on Applications of Signal Processing to Audio and Acoustics (WASPAA), pp. 45-49. IEEE, 2019.
</blockquote>

In [ ]:
# Mount Drive to work with dataset
# from google.colab import drive
# drive.mount('/content/drive/')

In [ ]:
# Location of the datset (using small subset for now)
data_home = './data'
dataset_name = 'slakh'
dataset_version = 'baby'
dataset = u.load_data(dataset_name, data_home=data_home, dataset_version=dataset_version)

# Run the following line once to download
# dataset.download()

# Download the index for mirdata to load
# dataset.download(partial_download=['index'])

# Uncomment the following line and run to validate
# dataset.validate()

### Sample a random multitrack and generate audio:

In [ ]:
# Sample a track and check Audio
example = dataset.choice_multitrack()
print(example.mtrack_id)

# listen to Dataset audio
Audio(example.audio[0], rate=example.audio[1])

# Listen to Synthesized Audio
synth = example.midi.synthesize(fs=44100)
Audio(synth, rate=44100)

## Step 1a: Segment the MIDI files by beats and downbeats:


In [ ]:
# Load all multitracks
data = dataset.load_multitracks()

# Get downbeats and beats for dataset using PrettyMIDI function
downbeats = u.beat_times(data, division='downbeat')
beats = u.beat_times(data, division='beat')

## Step 1b: Using beat and downbeat times, find segments of MIDI and store active notes

segment_midi() takes the dataset dictionary and the dictionary of beat times, storing an collection of values ('start', 'end', 'notes') for each segment of each track. All three dictionaries are keyed by mtrack_id

In [ ]:
# Get a dictionary of segment information for the dataset
beat_segments = u.segment_midi(data, beats)
dbeat_segments = u.segment_midi(data, downbeats)

### Create click track to check alignment of audio to midi segments?

In [ ]:
def create_click_waveform(freq, fs=44100, duration=0.05):
    t = np.linspace(0, duration, int(fs * duration))
    # generate sine wave
    click = np.sin(2 * np.pi * freq * t)
    # apply exponential decay (the envelope) so it sounds like a 'click'
    envelope = np.exp(-t / (0.01 * duration))
    return click * envelope

regular_waveform = create_click_waveform(440)

In [ ]:
audio = dataset.choice_multitrack()
track_id = audio.mtrack_id
audio_data, fs = audio.audio

beat_clicks = mir_eval.sonify.clicks(beats[track_id], fs=fs, length=len(audio_data), click=regular_waveform)
downbeat_clicks = mir_eval.sonify.clicks(downbeats[track_id], fs=fs, length=len(audio_data))
Audio(audio_data + beat_clicks + downbeat_clicks, rate=fs)

## Step 2a: Calculate the weighted Pitch Class Profile scores for each segment of each multitrack, relative to the velocity and duration of the notes active during the segment:

In [ ]:
beat_pcp = u.weighted_pitch_class(beat_segments)
dbeat_pcp = u.weighted_pitch_class(dbeat_segments)

## Step 2b: Define chord templates and compare Pitch Class Profile scores for each segment against all templates, finding the best chord estimate and similarity score

The CASSETTE paper used only major and minor triad templates, along with a No Chord template. At the moment, we have the same, but could be scaled up to a larger vocabulary in the following cell:

In [ ]:
# define chord templates
# start with basic maj/min + 'N' chords
CHORD_PATTERNS = {
    'maj': [1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
    'min': [1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0],
    'maj7': [1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1],
    'min7': [1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0], 
    '7': [1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0]
}

templates = u.get_all_templates(CHORD_PATTERNS)

In [ ]:
enharmonic_map = {
    'Db': 'C#',
    'Eb': 'D#',
    'Gb': 'F#',
    'Ab': 'G#',
    'Bb': 'A#',
    'Cb': 'B',
    'Fb': 'E',
    'E#': 'F',
    'B#': 'C'
}

<i>classify_chord()</i> from Utils.py uses function <i>calc_similarity()</i> to compare similarity scores for each template and store the best chord estimate + corresponding similarity score for each segment of a multitrack.

In [ ]:
# Create dictionaries to store estimates for all multitracks
# Each key will have chord array and similarity score array (n tracks x (2 x m segments))
beat_estimates = {}
dbeat_estimates = {}

for id, pcp in beat_pcp.items():
    beat_estimates[id] = u.classify_chord(pcp, templates)

for id, pcp in dbeat_pcp.items():
    dbeat_estimates[id] = u.classify_chord(pcp, templates)

In [ ]:
# Test that estimates are working
db_chordlen = len(dbeat_estimates['Track00016'][0])
db_simlen = len(dbeat_estimates['Track00016'][1])
b_chordlen = len(beat_estimates['Track00016'][0])
b_simlen = len(beat_estimates['Track00016'][1])

if  db_chordlen == db_simlen:
  print(f"Downbeat chords and similarities same length: {db_chordlen}")
if  b_chordlen == b_simlen:
  print(f"Beat chords and similarities same length: {b_chordlen}")

print(dbeat_estimates['Track00016'][0])
print(beat_estimates['Track00016'][0])

# Part C: Results and evaluation

FYI: chord estimates and respective similarity scores are stored in dictionaries <i>dbeat_estimates</i> and <i>beat_estimates</i> at indices [0] and [1], respectively.

In [ ]:
# get cassette outputs
for track in dataset.mtrack_ids:
    u.save_cassette_csv(beats, beat_estimates, track)

In [ ]:
# evaluate all 20 tracks
accuracy_list = []
accuracy_inv_list = []
accuracy_triad_list = []

for track in dataset.mtrack_ids:
    # load relevant cassette csv
    cassette_df = pd.read_csv(f'./output/cassette/{track}.csv')
    # load relevant crema csv
    crema_df = pd.read_csv(f'./output/crema/{track}.csv')

    # process crema outputs to align with cassette's for comparison
    crema_df = u.process_crema_df(crema_df, enharmonic_map)
    crema_beatwise_df = u.get_beatwise_crema(crema_df, cassette_df, u.majority_label)

    # manual evaluation
    accuracy, accuracy_no_inv, accuracy_triad = u.manual_chord_evaluation(crema_beatwise_df, cassette_df)
    accuracy_list.append(accuracy)
    accuracy_inv_list.append(accuracy_no_inv)
    accuracy_triad_list.append(accuracy_triad)

print(f"average accuracy: {np.mean(accuracy_list):.2f}%")
print(f'average accuracy without inversions: {np.mean(accuracy_list):.2f}%')
print(f"average accuracy triads: {np.mean(accuracy_triad_list):.2f}%")

#### mir_eval.chord.evaluate

In [ ]:
# without normalization
scores = {}

for track in dataset.mtrack_ids:

    # load relevant cassette csv
    cassette_df = pd.read_csv(f'./output/cassette/{track}.csv')
    # load relevant crema csv
    crema_df = pd.read_csv(f'./output/crema/{track}.csv')

    # process crema outputs to align with cassette's for comparison
    crema_df = u.process_crema_df(crema_df, enharmonic_map)
    
    # get chord scores
    score = u.get_mir_chord_scores(crema_df, cassette_df, normalize_labels=False)

    scores[track] = score

results_df = pd.DataFrame(scores).T
final_scores = results_df.mean(numeric_only=True)
print(final_scores)

In [ ]:
# with normalization
scores_normalized = {}

for track in dataset.mtrack_ids:

    # load relevant cassette csv
    cassette_df = pd.read_csv(f'./output/cassette/{track}.csv')
    # load relevant crema csv
    crema_df = pd.read_csv(f'./output/crema/{track}.csv')

    # process crema outputs to align with cassette's for comparison
    crema_df = u.process_crema_df(crema_df, enharmonic_map)
    
    # get chord scores
    score = u.get_mir_chord_scores(crema_df, cassette_df, normalize_labels=True)

    scores_normalized[track] = score

results_normalized_df = pd.DataFrame(scores_normalized).T
final_scores_normalized = results_normalized_df.mean(numeric_only=True)
print(final_scores_normalized)

both manual and mir_eval pipelines show same hierarchy of agreement: root-level similarity is high, triad-level similarity is moderate, and extension-sensitive metrics are lowest, regardless of normalization choices

In [ ]:
# plot cm for 1 track
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# load relevant cassette csv
cassette_df = pd.read_csv(f'./output/cassette/Track00001.csv')
# load relevant crema csv
crema_df = pd.read_csv(f'./output/crema/Track00001.csv')

# process crema outputs to align with cassette's for comparison
crema_df = u.process_crema_df(crema_df, enharmonic_map)
crema_beatwise_df = u.get_beatwise_crema(crema_df, cassette_df, u.majority_label)

y_true = crema_beatwise_df['value']
y_pred = cassette_df['value']
labels = sorted(set(y_true) | set(y_pred))

cm = confusion_matrix(y_true, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df.div(cm_df.sum(axis=1), axis=0) # normalize

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_df,
    cmap="Blues",
    annot=False,
    linewidths=0.5
)

plt.xlabel("Predicted (Cassette)")
plt.ylabel("True (CREMA)")
plt.title("Chord Confusion Matrix (Cassette vs CREMA)")
plt.tight_layout()
plt.show()